# 01 — Graph Representations, DFS, and BFS

## Why This Matters for DSA
Moving from trees to graphs is a major transition. In tree structures (which you completed in Phase 03), there is always exactly one path between any two nodes and cycles are strictly impossible. In general graphs, we lose these clean invariants: vertices can have multiple paths between them, cycles can cause infinite traversals, and components can be completely disconnected from one another. 

Understanding how to represent graphs in memory, how to traverse them deeply (DFS) or broadly (BFS), and how to identify properties like cycles or bipartiteness forms the essential bedrock for all advanced graph topics—including Dijkstra's, Bellman-Ford, Kruskal's, Prim's, and network flow.

## Prerequisites
- `08_Stacks_Queues_Deques` (Phase 02) — Queue operations for BFS and Stack concepts for DFS
- `03_Tree_DFS_BFS` (Phase 03) — Familiarity with DFS and BFS recursion and traversal shapes

## Learning Objectives
By the end of this notebook, you will be able to:
- Compare the time/space trade-offs of Adjacency Matrices, Adjacency Lists, and Edge Lists.
- Implement DFS on graphs to identify connected components and detect cycles in both directed and undirected graphs.
- Implement BFS to find shortest paths in unweighted networks.
- Model 2D grids as graphs and implement clean grid traversals using direction vectors.
- Check graph bipartiteness using two-coloring with BFS/DFS.
- Analyze the empirical performance of Adjacency Matrices vs. Adjacency Lists for lookup and neighbor iteration.

## Checklist Before Moving On
- [ ] I can explain the space complexity of an Adjacency Matrix ($O(V^2)$) vs. an Adjacency List ($O(V + E)$).
- [ ] I can write a directed cycle detection algorithm using 3-state (White/Gray/Black) coloring.
- [ ] I can write an undirected cycle detection algorithm and explain why the parent pointer is necessary.
- [ ] I can implement BFS shortest path on unweighted graphs and reconstruct the path from start to end.
- [ ] I can traverse a 2D grid using a directions array (`dx`/`dy`) and correct boundary validation.
- [ ] I can identify if a graph is bipartite and state the relation between bipartiteness and odd cycles.

## Table of Contents
1. Graph Representations — Matrix, List, and Implicit
2. Graph DFS — Cycle Detection and Connected Components
3. Graph BFS — Shortest Paths on Unweighted Graphs
4. Grid Traversals — Coordinates, Directions, and Boundaries
5. Graph Coloring — Bipartite Graphs
6. Benchmarking: Adjacency List vs Adjacency Matrix
7. Phase Checkpoint, Cheat Sheet, and Answer Key
8. LeetCode Practice Problems

## 1. Graph Representations — Matrix, List, and Implicit

### The Why
Before we can run any graph algorithms, we must store the graph in memory. How we represent nodes and edges directly dictates the time and space complexity of operations like adding an edge, checking if an edge exists, and iterating over all neighbors of a node. Choosing the wrong representation can turn an $O(V+E)$ algorithm into an $O(V^2)$ one, leading to TLE (Time Limit Exceeded) on large sparse graphs.

### Core Mechanism

We represent a graph $G = (V, E)$ containing $V$ vertices and $E$ edges in three main ways:

#### 1. Adjacency Matrix
An adjacency matrix is a 2D array of size $V \times V$, where `matrix[u][v]` holds a boolean value (or weight) representing whether there is an edge from $u$ to $v$.

```
   (0) --[w=5]-- (1)            Matrix representation:
    |             |               u \ v | 0 | 1 | 2 |
   [w=8]         [w=2]            ------+---+---+---+
    |             |                 0   | 0 | 5 | 8 |
   (2) -----------+                 1   | 5 | 0 | 2 |
                                    2   | 8 | 2 | 0 |
```

- **Space Complexity:** $O(V^2)$
- **Time Complexity:**
  - Check edge $(u, v)$: $O(1)$
  - Get all neighbors of $u$: $O(V)$
  - Add/Remove edge: $O(1)$
- **Best Use Case:** Dense graphs ($E \approx V^2$) or when $V$ is small ($V \le 2000$).

#### 2. Adjacency List
An adjacency list is an array of lists/vectors of size $V$, where `adj[u]` contains all adjacent neighbors of $u$ along with optional weights.

```
   Adjacency List representation:
   0: -> (1, w:5) -> (2, w:8)
   1: -> (0, w:5) -> (2, w:2)
   2: -> (0, w:8) -> (1, w:2)
```

- **Space Complexity:** $O(V + E)$
- **Time Complexity:**
  - Check edge $(u, v)$: $O(\text{deg}(u))$ (degree of $u$)
  - Get all neighbors of $u$: $O(\text{deg}(u))$
  - Add edge: $O(1)$
- **Best Use Case:** Sparse graphs ($E \ll V^2$). This is the standard representation for the vast majority of DSA problems.

#### 3. Edge List
An edge list stores all edges as a simple vector of tuples/structs: `vector<Edge>` where `Edge` contains `u`, `v`, and `weight`.
- **Space Complexity:** $O(E)$
- **Time Complexity:**
  - Check edge $(u, v)$: $O(E)$
  - Get neighbors: $O(E)$
- **Best Use Case:** Algorithms that sort edges or iterate through all edges sequentially (e.g., Kruskal's for Minimum Spanning Trees, Bellman-Ford for shortest paths).

#### 4. Implicit Graph
No storage is constructed! Neighbors are generated dynamically via transition functions (e.g. grid neighbors, legal moves in Chess, or strings differing by one character).

### Common Pitfalls
- **Memory Limit Exceeded (MLE):** Allocating a $V \times V$ adjacency matrix when $V = 10^5$. $10^{10}$ integers require 40 GB of RAM. Always default to an Adjacency List unless $V$ is very small.
- **Incorrectly handling undirected graphs:** When adding an edge $u \leftrightarrow v$, remember to add it *both* ways: `adj[u].push_back({v, weight})` and `adj[v].push_back({u, weight})`.

In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <vector>
#include <list>
#include <tuple>
#include <algorithm>

using namespace std;

// 1. ADJACENCY MATRIX REPRESENTATION (Dense Graphs)
class AdjMatrixGraph {
    int V;
    vector<vector<int>> matrix; // 1 for edge, 0 for no edge (or weight)
    bool isDirected;

public:
    AdjMatrixGraph(int vertices, bool directed = false) : V(vertices), isDirected(directed) {
        matrix.assign(V, vector<int>(V, 0));
    }

    void addEdge(int u, int v, int weight = 1) {
        matrix[u][v] = weight;
        if (!isDirected) {
            matrix[v][u] = weight;
        }
    }

    bool hasEdge(int u, int v) const {
        return matrix[u][v] != 0;
    }

    void print() const {
        cout << "Adjacency Matrix:\n";
        for (int i = 0; i < V; ++i) {
            for (int j = 0; j < V; ++j) {
                cout << matrix[i][j] << " ";
            }
            cout << "\n";
        }
    }
};

// 2. ADJACENCY LIST REPRESENTATION (Sparse Graphs)
class AdjListGraph {
    int V;
    vector<vector<pair<int, int>>> adj; // pair of {neighbor, weight}
    bool isDirected;

public:
    AdjListGraph(int vertices, bool directed = false) : V(vertices), isDirected(directed) {
        adj.resize(V);
    }

    void addEdge(int u, int v, int weight = 1) {
        adj[u].push_back({v, weight});
        if (!isDirected) {
            adj[v].push_back({u, weight});
        }
    }

    bool hasEdge(int u, int v) const {
        for (const auto& edge : adj[u]) {
            if (edge.first == v) return true;
        }
        return false;
    }

    void print() const {
        cout << "Adjacency List:\n";
        for (int i = 0; i < V; ++i) {
            cout << i << ": ";
            for (const auto& edge : adj[i]) {
                cout << "-> (" << edge.first << ", w:" << edge.second << ") ";
            }
            cout << "\n";
        }
    }
};

// 3. EDGE LIST REPRESENTATION (Kruskal's, Bellman-Ford)
struct Edge {
    int u, v, weight;
};

int main() {
    // Let's build a simple graph: 4 nodes, undirected
    // 0 - 1 (w=5)
    // 0 - 2 (w=8)
    // 1 - 2 (w=2)
    // 2 - 3 (w=3)
    
    cout << "--- Adjacency Matrix ---\n";
    AdjMatrixGraph matGraph(4, false);
    matGraph.addEdge(0, 1, 5);
    matGraph.addEdge(0, 2, 8);
    matGraph.addEdge(1, 2, 2);
    matGraph.addEdge(2, 3, 3);
    matGraph.print();
    cout << "Has edge 0-2? " << (matGraph.hasEdge(0, 2) ? "Yes" : "No") << "\n";
    cout << "Has edge 1-3? " << (matGraph.hasEdge(1, 3) ? "Yes" : "No") << "\n\n";

    cout << "--- Adjacency List ---\n";
    AdjListGraph listGraph(4, false);
    listGraph.addEdge(0, 1, 5);
    listGraph.addEdge(0, 2, 8);
    listGraph.addEdge(1, 2, 2);
    listGraph.addEdge(2, 3, 3);
    listGraph.print();
    cout << "Has edge 0-2? " << (listGraph.hasEdge(0, 2) ? "Yes" : "No") << "\n";
    cout << "Has edge 1-3? " << (listGraph.hasEdge(1, 3) ? "Yes" : "No") << "\n\n";

    cout << "--- Edge List ---\n";
    vector<Edge> edgeList = {
        {0, 1, 5},
        {0, 2, 8},
        {1, 2, 2},
        {2, 3, 3}
    };
    for (const auto& edge : edgeList) {
        cout << "Edge: " << edge.u << " <-> " << edge.v << " (weight: " << edge.weight << ")\n";
    }

    return 0;
}

In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 2. Graph DFS — Cycle Detection and Connected Components

### The Why
Depth-First Search (DFS) is the standard tool for exploring paths, connectivity, and cycle structures. Unlike Tree DFS, graphs can have cycles. If we recursively visit neighbors without tracking where we have already been, we will fall into an infinite loop and crash with a Stack Overflow.

### Core Mechanism

#### 1. Connected Components (Undirected Graphs)
If a graph is disconnected (made of isolated subgraphs), running DFS from a single node will not visit the entire graph. We must iterate from $0$ to $V-1$ and start a new DFS whenever we encounter an unvisited node. Each start represents a distinct connected component.

#### 2. Directed Cycle Detection (3-State Coloring)
Using a simple binary `visited` array is not enough to detect cycles in a directed graph. In a Directed Acyclic Graph (DAG), you can reach a node via multiple paths (e.g. cross edges) without there being a cycle:
```
  0 -> 1 -> 3
  0 -> 2 -> 3
```
If we run DFS: `0 -> 1 -> 3` (marks 3 visited). Later, `0 -> 2 -> 3` sees 3 visited. If we assume "visited node = cycle", we would falsely report a cycle. 

To solve this, we use **3-state coloring**:
- **White (0):** Unvisited node.
- **Gray (1):** Currently visiting. The node is in the current recursion stack (ancestor).
- **Black (2):** Fully processed. The node and all its descendants have been visited and we backtracked.
*A cycle exists if and only if we encounter a neighbor that is currently in the **Gray (1)** state.* This means we found a **back-edge** to an ancestor in the active call stack.

#### 3. Undirected Cycle Detection
In an undirected graph, every edge is bidirectional. If we are at node $u$ and visit neighbor $v$, $v$ will see $u$ as a neighbor. We must not mistake this immediate return trip as a cycle. 
To prevent this, we pass a `parent` parameter in our DFS. If $v$ is already visited and $v \neq parent$, then we have found a cycle.

### Common Pitfalls
- **Directed vs. Undirected cycle logic mismatch:** Using the `parent` check for directed graphs (fails because edges are one-way) or using simple 3-state coloring on undirected graphs without ignoring the direct parent (mistakes the undirected edge as a cycle).

In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <vector>
#include <string>

using namespace std;

class DFSGraph {
    int V;
    vector<vector<int>> adj;
    bool isDirected;

public:
    DFSGraph(int vertices, bool directed = false) : V(vertices), isDirected(directed) {
        adj.resize(V);
    }

    void addEdge(int u, int v) {
        adj[u].push_back(v);
        if (!isDirected) {
            adj[v].push_back(u);
        }
    }

    // Helper for finding connected components in undirected graph
    void dfsComponent(int u, vector<bool>& visited, vector<int>& component) {
        visited[u] = true;
        component.push_back(u);
        for (int v : adj[u]) {
            if (!visited[v]) {
                dfsComponent(v, visited, component);
            }
        }
    }

    // Connected components interface
    vector<vector<int>> findConnectedComponents() {
        vector<bool> visited(V, false);
        vector<vector<int>> components;
        for (int i = 0; i < V; ++i) {
            if (!visited[i]) {
                vector<int> component;
                dfsComponent(i, visited, component);
                components.push_back(component);
            }
        }
        return components;
    }

    // Directed cycle detection using 3-state coloring
    // 0: Unvisited (White)
    // 1: Visiting (Gray - in recursion stack)
    // 2: Visited (Black - completely finished)
    bool hasDirectedCycleHelper(int u, vector<int>& state) {
        state[u] = 1; // Mark as Gray (currently visiting)

        for (int v : adj[u]) {
            if (state[v] == 1) {
                return true; // Found a back edge (cycle detected)
            }
            if (state[v] == 0) {
                if (hasDirectedCycleHelper(v, state)) {
                    return true;
                }
            }
        }

        state[u] = 2; // Mark as Black (fully processed)
        return false;
    }

    bool hasDirectedCycle() {
        vector<int> state(V, 0); // Initialize all to White
        for (int i = 0; i < V; ++i) {
            if (state[i] == 0) {
                if (hasDirectedCycleHelper(i, state)) {
                    return true;
                }
            }
        }
        return false;
    }

    // Undirected cycle detection using parent tracking
    bool hasUndirectedCycleHelper(int u, int parent, vector<bool>& visited) {
        visited[u] = true;

        for (int v : adj[u]) {
            if (!visited[v]) {
                if (hasUndirectedCycleHelper(v, u, visited)) {
                    return true;
                }
            } else if (v != parent) {
                // If neighbor is visited and not the parent, a cycle exists
                return true;
            }
        }
        return false;
    }

    bool hasUndirectedCycle() {
        vector<bool> visited(V, false);
        for (int i = 0; i < V; ++i) {
            if (!visited[i]) {
                if (hasUndirectedCycleHelper(i, -1, visited)) {
                    return true;
                }
            }
        }
        return false;
    }
};

int main() {
    // 1. Undirected Connected Components
    cout << "--- Connected Components on Undirected Graph ---\n";
    DFSGraph g1(6, false);
    g1.addEdge(0, 1);
    g1.addEdge(1, 2);
    g1.addEdge(3, 4);
    // Node 5 is isolated
    auto components = g1.findConnectedComponents();
    for (int i = 0; i < (int)components.size(); ++i) {
        cout << "Component " << i + 1 << ": ";
        for (int node : components[i]) cout << node << " ";
        cout << "\n";
    }

    // 2. Undirected Cycle Detection
    cout << "\n--- Undirected Cycle Detection ---\n";
    DFSGraph g2(4, false);
    g2.addEdge(0, 1);
    g2.addEdge(1, 2);
    g2.addEdge(2, 3);
    cout << "Linear graph has cycle? " << (g2.hasUndirectedCycle() ? "Yes" : "No") << "\n";
    g2.addEdge(3, 0); // add closing edge
    cout << "Cyclic graph has cycle? " << (g2.hasUndirectedCycle() ? "Yes" : "No") << "\n";

    // 3. Directed Cycle Detection (3-State Coloring)
    cout << "\n--- Directed Cycle Detection ---\n";
    DFSGraph g3(4, true); // Directed DAG
    g3.addEdge(0, 1);
    g3.addEdge(0, 2);
    g3.addEdge(1, 3);
    g3.addEdge(2, 3); // Cross edges exist, but no cycle
    cout << "DAG has cycle? " << (g3.hasDirectedCycle() ? "Yes" : "No") << "\n";
    
    DFSGraph g4(4, true); // Directed Cyclic
    g4.addEdge(0, 1);
    g4.addEdge(1, 2);
    g4.addEdge(2, 0); // cycle 0 -> 1 -> 2 -> 0
    g4.addEdge(2, 3);
    cout << "Cyclic directed graph has cycle? " << (g4.hasDirectedCycle() ? "Yes" : "No") << "\n";

    return 0;
}

In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 3. Graph BFS — Shortest Paths on Unweighted Graphs

### The Why
Breadth-First Search (BFS) explores the graph level-by-level, expanding outward like a ripple in water. Because of this structural property, BFS guarantees finding the path with the *minimum number of edges* (shortest path) from a starting vertex to all other vertices in an unweighted graph.

### Core Mechanism
1. **Queue:** Use a FIFO queue to store nodes waiting to be expanded.
2. **Distance Array:** Maintain a `dist` array initialized to `-1` (meaning unreachable/infinity). The starting node has `dist[start] = 0`.
3. **Parent Array:** To reconstruct the shortest path, keep a `parent` array where `parent[v] = u` when we discover node $v$ from $u$.
4. **Queue Invariant (Marking Visited):** Mark a node as visited (by setting its distance or using a `visited` flag) **at the exact moment you push it into the queue**, not when you pop it.

```
       (Start: 0, dist=0)
          /        \
   (1, dist=1)     (3, dist=1)
        |               |
   (2, dist=2)     (4, dist=2)
        \               /
       (5, dist=3, shortest)
```

### Common Pitfalls
- **Pop-Time Visited Marking (Critical TLE/MLE Bug):** If you mark a node as visited only when it is popped from the queue, a node can be pushed into the queue multiple times by different neighbors before it gets popped. On dense or grid graphs, this causes the queue size to explode exponentially, leading to Time Limit Exceeded (TLE) or Memory Limit Exceeded (MLE).
  *Correction:* Set `visited[v] = true` (or `dist[v] = dist[u] + 1`) **immediately** when pushing $v$ to the queue.

In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <vector>
#include <queue>
#include <algorithm>

using namespace std;

class BFSGraph {
    int V;
    vector<vector<int>> adj;

public:
    BFSGraph(int vertices) : V(vertices) {
        adj.resize(V);
    }

    void addEdge(int u, int v, bool directed = false) {
        adj[u].push_back(v);
        if (!directed) {
            adj[v].push_back(u);
        }
    }

    // BFS shortest path on unweighted graph
    // Returns {distances, parent array}
    pair<vector<int>, vector<int>> bfsShortestPath(int start) {
        vector<int> dist(V, -1);      // -1 represents infinity/unreachable
        vector<int> parent(V, -1);    // to reconstruct paths
        queue<int> q;

        dist[start] = 0;
        q.push(start); // Mark as visited immediately upon insertion (dist[start]=0 serves as visited flag)

        while (!q.empty()) {
            int u = q.front();
            q.pop();

            for (int v : adj[u]) {
                if (dist[v] == -1) {  // If unvisited
                    dist[v] = dist[u] + 1;
                    parent[v] = u;
                    q.push(v); // Mark visited immediately here to avoid duplicates
                }
            }
        }
        return {dist, parent};
    }

    // Path reconstruction helper
    vector<int> reconstructPath(int start, int end, const vector<int>& parent) {
        vector<int> path;
        if (parent[end] == -1 && start != end) {
            return path; // No path exists
        }
        for (int at = end; at != -1; at = parent[at]) {
            path.push_back(at);
        }
        reverse(path.begin(), path.end());
        return path;
    }
};

int main() {
    // Create an unweighted graph:
    // 0 - 1 - 2
    // |       |
    // 3 - 4 - 5
    // Path from 0 to 2 is 0-1-2 (length 2)
    // Path from 0 to 5 is 0-3-4-5 (length 3) or 0-1-2-5 (length 3 if 2-5 exists)
    
    BFSGraph g(6);
    g.addEdge(0, 1);
    g.addEdge(1, 2);
    g.addEdge(0, 3);
    g.addEdge(3, 4);
    g.addEdge(4, 5);
    g.addEdge(2, 5); // adding link 2-5

    int startNode = 0;
    auto [dist, parent] = g.bfsShortestPath(startNode);

    cout << "--- BFS Shortest Distances from Node " << startNode << " ---\n";
    for (int i = 0; i < 6; ++i) {
        cout << "To Node " << i << ": Distance = " << dist[i] << "\n";
    }

    cout << "\n--- Path Reconstruction ---\n";
    int targetNode = 5;
    auto path = g.reconstructPath(startNode, targetNode, parent);
    cout << "Shortest Path " << startNode << " -> " << targetNode << ": ";
    for (int node : path) {
        cout << node << " ";
    }
    cout << "\n";

    return 0;
}

In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 4. Grid Traversals — Coordinates, Directions, and Boundaries

### The Why
In LeetCode and competitive programming, graphs are frequently represented implicitly as 2D grids (matrices of cells). Each cell `(r, c)` is a vertex, and its neighbors are the adjacent cells (Up, Down, Left, Right, and sometimes diagonals). Building an explicit adjacency list for a grid is slow and wastes memory; instead, we traverse the grid directly using coordinates.

### Core Mechanism

#### 1. Directions Vectors
Rather than writing four separate blocks of code for Up, Down, Left, and Right, we define direction offset arrays:
```cpp
const int dx[] = {-1, 1, 0, 0}; // Row offsets
const int dy[] = {0, 0, -1, 1}; // Col offsets
```
We can then loop from $0$ to $3$ to calculate neighbor coordinates:
```cpp
for (int i = 0; i < 4; ++i) {
    int nr = r + dx[i];
    int nc = c + dy[i];
}
```

#### 2. Boundary Checking
Every neighbor coordinate `(nr, nc)` must be validated to ensure it falls within the grid boundaries before accessing the matrix:
```cpp
bool isValid(int r, int c, int R, int C) {
    return r >= 0 && r < R && c >= 0 && c < C;
}
```

### Common Pitfalls
- **Out of Bounds accesses:** Accessing `grid[nr][nc]` *before* calling `isValid(nr, nc)`. Ensure boundary checks are executed first in your logical statements.
- **Copy-Paste Bugs:** Writing four individual code blocks for neighbors (`r+1`, `r-1`, `c+1`, `c-1`) instead of a loop with direction vectors. This is the single most common source of coordinate typos.

In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <vector>
#include <queue>
#include <string>

using namespace std;

// Direction arrays for 4-way movement (Up, Down, Left, Right)
const int dx[] = {-1, 1, 0, 0};
const int dy[] = {0, 0, -1, 1};

// Helper function to check if coordinate is inside grid boundaries
bool isValid(int r, int c, int R, int C) {
    return r >= 0 && r < R && c >= 0 && c < C;
}

// 1. Grid DFS: Counting Islands (LeetCode 200)
void dfsGrid(int r, int c, vector<vector<char>>& grid, vector<vector<bool>>& visited) {
    int R = grid.size();
    int C = grid[0].size();
    visited[r][c] = true;

    for (int i = 0; i < 4; ++i) {
        int nr = r + dx[i];
        int nc = c + dy[i];

        // Recurse if neighbor is valid, is land ('1'), and not visited
        if (isValid(nr, nc, R, C) && grid[nr][nc] == '1' && !visited[nr][nc]) {
            dfsGrid(nr, nc, grid, visited);
        }
    }
}

int countIslands(vector<vector<char>>& grid) {
    if (grid.empty()) return 0;
    int R = grid.size();
    int C = grid[0].size();
    vector<vector<bool>> visited(R, vector<bool>(C, false));
    int islandCount = 0;

    for (int r = 0; r < R; ++r) {
        for (int c = 0; c < C; ++c) {
            if (grid[r][c] == '1' && !visited[r][c]) {
                islandCount++;
                dfsGrid(r, c, grid, visited); // Flood fill this island
            }
        }
    }
    return islandCount;
}

// 2. Grid BFS: Shortest Path in a Maze
// Returns length of shortest path from (0,0) to (R-1, C-1) or -1 if blocked
int shortestPathInMaze(const vector<vector<int>>& maze) {
    int R = maze.size();
    int C = maze[0].size();
    if (maze[0][0] == 1 || maze[R-1][C-1] == 1) return -1; // Blocked at start or end

    vector<vector<int>> dist(R, vector<int>(C, -1));
    queue<pair<int, int>> q;

    dist[0][0] = 0;
    q.push({0, 0});

    while (!q.empty()) {
        auto [r, c] = q.front();
        q.pop();

        if (r == R - 1 && c == C - 1) {
            return dist[r][c];
        }

        for (int i = 0; i < 4; ++i) {
            int nr = r + dx[i];
            int nc = c + dy[i];

            // Maze cell is 0 for free path, 1 for wall
            if (isValid(nr, nc, R, C) && maze[nr][nc] == 0 && dist[nr][nc] == -1) {
                dist[nr][nc] = dist[r][c] + 1;
                q.push({nr, nc});
            }
        }
    }
    return -1;
}

int main() {
    // Test counting islands
    vector<vector<char>> grid = {
        {'1', '1', '0', '0', '0'},
        {'1', '1', '0', '0', '0'},
        {'0', '0', '1', '0', '0'},
        {'0', '0', '0', '1', '1'}
    };
    cout << "Number of Islands: " << countIslands(grid) << "\n"; // Expected: 3

    // Test shortest path in a binary maze (0 = path, 1 = wall)
    vector<vector<int>> maze = {
        {0, 0, 1, 0},
        {1, 0, 0, 1},
        {0, 1, 0, 0},
        {0, 0, 0, 0}
    };
    cout << "Shortest path length in maze: " << shortestPathInMaze(maze) << "\n"; // Expected: 6 steps (0,0 -> 0,1 -> 1,1 -> 1,2 -> 2,2 -> 2,3 -> 3,3)

    return 0;
}

In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 5. Graph Coloring — Bipartite Graphs

### The Why
A bipartite graph (or bigraph) is a graph whose vertices can be divided into two disjoint sets $U$ and $V$ such that every edge connects a vertex in $U$ to one in $V$. No edges can exist between vertices in the same set. Bipartite graphs are crucial for matching problems (e.g., job-to-candidate assignments) and scheduling.

```
       Set A          Set B
        (0) ---------> (1)
        (2) ---------> (3)
         \            /
          `---------> (4)
   No edges exist within Set A or Set B.
```

### Core Mechanism
A graph is bipartite if and only if it contains **no cycles of odd length**. We can check this by trying to color the graph using 2 colors (e.g., 0 and 1) such that no two adjacent nodes have the same color:
1. Start with an uncolored graph (color array initialized to `-1`).
2. For each unvisited component, color the starting node with `0`.
3. Perform BFS or DFS. For each neighbor:
   - If the neighbor is uncolored, assign it the opposite color: `color[neighbor] = 1 - color[current]`, and push to queue/recurse.
   - If the neighbor is already colored, check if its color matches the current node's color. If they are the same, the graph contains an odd cycle and is **not bipartite**.

### Common Pitfalls
- **Ignoring Disconnected Components:** Only running the coloring algorithm from node `0`. If the graph has multiple components, some vertices will remain uncolored, and odd cycles in other components will go undetected. Always loop through all vertices to check for uncolored nodes.

In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <vector>
#include <queue>

using namespace std;

class BipartiteGraph {
    int V;
    vector<vector<int>> adj;

public:
    BipartiteGraph(int vertices) : V(vertices) {
        adj.resize(V);
    }

    void addEdge(int u, int v) {
        adj[u].push_back(v);
        adj[v].push_back(u);
    }

    // Bipartite check using BFS coloring
    // Colors: -1 (Uncolored), 0 (Color A), 1 (Color B)
    bool isBipartite() {
        vector<int> color(V, -1);

        for (int i = 0; i < V; ++i) {
            if (color[i] == -1) { // Process each connected component
                queue<int> q;
                color[i] = 0; // Start coloring component with Color 0
                q.push(i);

                while (!q.empty()) {
                    int u = q.front();
                    q.pop();

                    for (int v : adj[u]) {
                        if (color[v] == -1) {
                            // Assign opposite color
                            color[v] = 1 - color[u];
                            q.push(v);
                        } else if (color[v] == color[u]) {
                            // Adjacent nodes have same color -> Odd cycle exists -> Not bipartite
                            return false;
                        }
                    }
                }
            }
        }
        return true;
    }
};

int main() {
    // Bipartite Graph test (even length cycles or trees, e.g., a square 4-node ring)
    // 0 - 1
    // |   |
    // 3 - 2
    cout << "--- Testing Bipartite Graph (Square Ring) ---\n";
    BipartiteGraph g1(4);
    g1.addEdge(0, 1);
    g1.addEdge(1, 2);
    g1.addEdge(2, 3);
    g1.addEdge(3, 0);
    cout << "Is g1 Bipartite? " << (g1.isBipartite() ? "Yes" : "No") << "\n"; // Expected: Yes

    // Non-Bipartite Graph test (contains odd cycle, e.g., a triangle 3-node ring)
    // 0 - 1
    //  \ /
    //   2
    cout << "\n--- Testing Non-Bipartite Graph (Triangle Ring) ---\n";
    BipartiteGraph g2(3);
    g2.addEdge(0, 1);
    g2.addEdge(1, 2);
    g2.addEdge(2, 0);
    cout << "Is g2 Bipartite? " << (g2.isBipartite() ? "Yes" : "No") << "\n"; // Expected: No

    return 0;
}

In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 6. Benchmarking: Adjacency List vs Adjacency Matrix

### The Why
Asymptotic complexity is a powerful guide, but understanding the actual physical costs (cache locality, memory overhead, and search times) on real hardware is what separates theoreticians from systems-level engineers. This section provides an empirical C++ benchmark to measure these differences.

### Core Mechanism
We create a sparse graph ($V = 5000, E = 10000$) and run operations using both representations:
1. **Memory Allocation:** Compare memory footprints.
2. **Neighbor Iteration:** Measure the time required to visit every neighbor of every vertex (essential for DFS/BFS).
3. **Edge Existence Query:** Measure the time required to check if a random edge $(u, v)$ exists.

In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <vector>
#include <chrono>
#include <random>
#include <numeric>

using namespace std;

// Class to measure elapsed time in microseconds
struct Timer {
    string name;
    chrono::high_resolution_clock::time_point start;

    Timer(const string& n) : name(n), start(chrono::high_resolution_clock::now()) {}

    ~Timer() {
        auto end = chrono::high_resolution_clock::now();
        auto diff = chrono::duration_cast<chrono::microseconds>(end - start).count();
        cout << name << ": " << diff << " microseconds\n";
    }
};

int main() {
    const int V = 5000;
    const int E = 10000; // Sparse graph

    // Prepare random edges
    vector<pair<int, int>> edges;
    mt19937 rng(42);
    uniform_int_distribution<int> distV(0, V - 1);

    while ((int)edges.size() < E) {
        int u = distV(rng);
        int v = distV(rng);
        if (u != v) {
            edges.push_back({u, v});
        }
    }

    // 1. Build Adjacency Matrix
    vector<vector<int>> adjMat(V, vector<int>(V, 0));
    {
        Timer t("Build Adjacency Matrix");
        for (const auto& edge : edges) {
            adjMat[edge.first][edge.second] = 1;
            adjMat[edge.second][edge.first] = 1;
        }
    }

    // 2. Build Adjacency List
    vector<vector<int>> adjList(V);
    {
        Timer t("Build Adjacency List");
        for (const auto& edge : edges) {
            adjList[edge.first].push_back(edge.second);
            adjList[edge.second].push_back(edge.first);
        }
    }

    // 3. Benchmark neighbor iteration: Visit all neighbors of all vertices
    long long listSum = 0;
    {
        Timer t("Iterate Neighbors (Adjacency List)");
        for (int i = 0; i < V; ++i) {
            for (int neighbor : adjList[i]) {
                listSum += neighbor;
            }
        }
    }

    long long matSum = 0;
    {
        Timer t("Iterate Neighbors (Adjacency Matrix)");
        for (int i = 0; i < V; ++i) {
            for (int j = 0; j < V; ++j) {
                if (adjMat[i][j]) {
                    matSum += j;
                }
            }
        }
    }

    // 4. Benchmark edge existence lookup: Check 50,000 random potential edges
    const int lookups = 50000;
    vector<pair<int, int>> queryEdges;
    for (int i = 0; i < lookups; ++i) {
        queryEdges.push_back({distV(rng), distV(rng)});
    }

    int foundList = 0;
    {
        Timer t("Edge Lookup (Adjacency List)");
        for (const auto& q : queryEdges) {
            int u = q.first;
            int v = q.second;
            // Linear search in adjList[u]
            for (int neighbor : adjList[u]) {
                if (neighbor == v) {
                    foundList++;
                    break;
                }
            }
        }
    }

    int foundMat = 0;
    {
        Timer t("Edge Lookup (Adjacency Matrix)");
        for (const auto& q : queryEdges) {
            if (adjMat[q.first][q.second]) {
                foundMat++;
            }
        }
    }

    cout << "\nResults Check:\n";
    cout << "Neighbor sum check: List = " << listSum << ", Matrix = " << matSum << "\n";
    cout << "Edge found count: List = " << foundList << ", Matrix = " << foundMat << "\n";
    cout << "Memory consumption:\n";
    cout << "Adjacency Matrix: ~" << (V * V * sizeof(int)) / (1024 * 1024) << " MB\n";
    cout << "Adjacency List:   ~" << (V * sizeof(vector<int>) + E * 2 * sizeof(int)) / 1024 << " KB\n";

    return 0;
}

In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 7. Phase Checkpoint, Cheat Sheet, and Answer Key

### Graph Representation & Traversal Cheat Sheet

| Representation / Traversal | Space Complexity | Time Complexity | Ideal Use Case |
|---|---|---|---|
| **Adjacency Matrix** | $O(V^2)$ | Edge query: $O(1)$<br>Find neighbors: $O(V)$ | Dense graphs ($E \approx V^2$), small $V$ ($V \le 2000$). |
| **Adjacency List** | $O(V + E)$ | Edge query: $O(\text{deg}(u))$<br>Find neighbors: $O(\text{deg}(u))$ | Sparse graphs ($E \ll V^2$). Standard representation. |
| **Edge List** | $O(E)$ | Edge query: $O(E)$<br>Find neighbors: $O(E)$ | Sorting edges, e.g., Kruskal's MST, Bellman-Ford. |
| **DFS Traversal** | $O(V + E)$ | $O(V + E)$ | Pathfinding, cycle detection, connected components. |
| **BFS Traversal** | $O(V + E)$ | $O(V + E)$ | Shortest path on unweighted graphs. |

### Checkpoint Questions
1. Why does cycle detection in Directed graphs require 3 states (White/Gray/Black), whereas Undirected graphs only require parent tracking?
2. What happens to the queue size if you mark a vertex as visited when popping it from the queue in BFS, rather than when pushing it?
3. When is an Adjacency Matrix preferred over an Adjacency List?
4. What is the time complexity of checking if a graph contains a cycle using DFS?
5. How can we check if a graph is bipartite using BFS?
6. When representing grids as graphs, why do we use directions arrays (`dx`/`dy`) instead of explicit adjacency lists?

### Answer Key
1. Directed graphs contain one-way edges. A vertex can be visited multiple times from different paths without forming a cycle (e.g. cross edges in DAGs). Thus, we must distinguish between visiting a node currently in the active stack (Gray) and one already fully explored (Black). In undirected graphs, edges are bidirectional, so a cycle only occurs if we reach an already visited node that is not our direct parent.
2. The queue size will explode (possibly leading to MLE or TLE). Multiple neighbor nodes can push the same unvisited node into the queue before it is popped and marked visited.
3. When the graph is dense ($E \approx V^2$), or when we require instant $O(1)$ edge existence checks and have small $V$.
4. $O(V + E)$, because DFS visits every vertex and explores every edge exactly once.
5. Try to color the graph with 2 colors. If any adjacent nodes share the same color, the graph is not bipartite. If we can color the whole graph without conflict, it is bipartite.
6. Grids have a fixed maximum degree of 4 (or 8 for diagonals). Using coordinates and simple offset vectors is faster, requires $O(1)$ auxiliary code, and uses zero additional memory compared to building and storing an adjacency list.

### Mini Project: Flight Network Simulator
Implement a flight database using adjacency lists. Write functions to:
1. Detect whether there are loops in the flight schedules (Directed Cycle Detection).
2. Calculate the minimum layovers needed between any two airports (BFS Shortest Path).
3. Determine if the airports can be partitioned into two distinct groups such that flights only occur between groups (Bipartite Graph Coloring).

In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <vector>
#include <queue>
#include <string>
#include <algorithm>
#include <unordered_map>

using namespace std;

class FlightNetwork {
    int V;
    vector<vector<int>> adj; // adjacency list for airports
    unordered_map<int, string> idToName;
    unordered_map<string, int> nameToId;

public:
    FlightNetwork(const vector<string>& airports) {
        V = airports.size();
        adj.resize(V);
        for (int i = 0; i < V; ++i) {
            idToName[i] = airports[i];
            nameToId[airports[i]] = i;
        }
    }

    void addFlight(const string& from, const string& to) {
        int u = nameToId[from];
        int v = nameToId[to];
        adj[u].push_back(v); // directed flight
    }

    // 1. BFS to find minimum number of layovers
    int getMinLayovers(const string& start, const string& dest) {
        int startId = nameToId[start];
        int destId = nameToId[dest];

        vector<int> dist(V, -1);
        queue<int> q;

        dist[startId] = 0;
        q.push(startId);

        while (!q.empty()) {
            int u = q.front();
            q.pop();

            if (u == destId) {
                return dist[u] - 1; // Number of layovers is edges - 1
            }

            for (int v : adj[u]) {
                if (dist[v] == -1) {
                    dist[v] = dist[u] + 1;
                    q.push(v);
                }
            }
        }
        return -1; // Unreachable
    }

    // 2. DFS for cycle detection (detect loops in flight schedules)
    bool hasCycleHelper(int u, vector<int>& state) {
        state[u] = 1; // Visiting (Gray)
        for (int v : adj[u]) {
            if (state[v] == 1) return true; // Cycle detected!
            if (state[v] == 0) {
                if (hasCycleHelper(v, state)) return true;
            }
        }
        state[u] = 2; // Visited (Black)
        return false;
    }

    bool hasCycle() {
        vector<int> state(V, 0);
        for (int i = 0; i < V; ++i) {
            if (state[i] == 0) {
                if (hasCycleHelper(i, state)) return true;
            }
        }
        return false;
    }

    // 3. Bipartite flight categorization check
    // Can we split airports into two sets such that all flights only connect Set A -> Set B?
    // Note: Since flights are directed, Bipartite check on underlying undirected graph is standard.
    // For this demonstration, we'll treat it as undirected for coloring.
    bool canCategorizeAirports() {
        // Build undirected version for bipartite check
        vector<vector<int>> undirectedAdj(V);
        for (int u = 0; u < V; ++u) {
            for (int v : adj[u]) {
                undirectedAdj[u].push_back(v);
                undirectedAdj[v].push_back(u);
            }
        }

        vector<int> color(V, -1);
        for (int i = 0; i < V; ++i) {
            if (color[i] == -1) {
                queue<int> q;
                color[i] = 0;
                q.push(i);

                while (!q.empty()) {
                    int u = q.front();
                    q.pop();

                    for (int v : undirectedAdj[u]) {
                        if (color[v] == -1) {
                            color[v] = 1 - color[u];
                            q.push(v);
                        } else if (color[v] == color[u]) {
                            return false;
                        }
                    }
                }
            }
        }
        return true;
    }
};

int main() {
    vector<string> airports = {"LHR", "JFK", "DXB", "SIN", "HND", "CDG"};
    FlightNetwork network(airports);

    // Add flights
    network.addFlight("LHR", "JFK");
    network.addFlight("JFK", "DXB");
    network.addFlight("DXB", "SIN");
    network.addFlight("SIN", "HND");
    network.addFlight("HND", "CDG");

    cout << "--- Flight Connections Simulation ---\n";
    cout << "Min layovers from LHR to HND: " << network.getMinLayovers("LHR", "HND") << "\n"; // Expected: 3 layovers (LHR -> JFK -> DXB -> SIN -> HND)
    cout << "Min layovers from DXB to CDG: " << network.getMinLayovers("DXB", "CDG") << "\n"; // Expected: 2 layovers (DXB -> SIN -> HND -> CDG)
    cout << "Is flight path cyclic? " << (network.hasCycle() ? "Yes" : "No") << "\n"; // Expected: No

    // Add a cycle: HND -> LHR
    cout << "\nAdding flight: HND -> LHR\n";
    network.addFlight("HND", "LHR");
    cout << "Is flight path cyclic now? " << (network.hasCycle() ? "Yes" : "No") << "\n"; // Expected: Yes

    // Check bipartite
    cout << "\nCan airports be color-coded into two sets with no same-set flights? ";
    cout << (network.canCategorizeAirports() ? "Yes" : "No") << "\n"; // LHR-JFK-DXB-SIN-HND-LHR is odd length cycle (5), so No

    return 0;
}

In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 8. LeetCode Practice Problems

| # | Problem | Difficulty | Topic | Hint |
|---|---|---|---|---|
| 1971 | Find if Path Exists in Graph | Easy | BFS / DFS | Simple reachability; start a traversal from the source node and check if the destination is marked visited. |
| 797 | All Paths From Source to Target | Medium | DFS Backtracking | Find all paths in a DAG; use DFS backtracking, adding nodes to a path list and popping them on backtrack. |
| 200 | Number of Islands | Medium | Grid DFS / BFS | Treat the grid as a graph of land cells. Run DFS/BFS from each unvisited land cell to mark its entire island. |
| 994 | Rotting Oranges | Medium | Multisource BFS | Initialize BFS queue with *all* rotten oranges at time 0. Process level-by-level to simulate simultaneous decay propagation. |
| 547 | Number of Provinces | Medium | Connected Components | Count connected components in an undirected graph represented by an adjacency matrix. |
| 785 | Is Graph Bipartite? | Medium | Graph Coloring | Perform 2-coloring using BFS/DFS. Check for color conflicts on already-visited neighbors. |
| 133 | Clone Graph | Medium | DFS / BFS Map | Traverse the original graph, using a hash map to map original nodes to their cloned copies to handle cycles correctly. |
| 1020 | Number of Enclaves | Medium | Grid DFS Boundary | Run DFS starting from all boundary land cells to mark connected land cells. Count the remaining unmarked land cells. |
| 841 | Keys and Rooms | Medium | Reachability | Rooms are vertices, keys are directed edges. Run BFS/DFS from room 0 and check if all rooms are visited. |
| 207 | Course Schedule | Medium | Directed Cycle | Detect cycles in a directed graph. Map courses to prerequisites and run 3-state cycle detection. |

### Self-Check Before Moving to `02_Topological_Sort_and_DSU`
- Can you explain directed cycle detection without hesitation?
- Do you understand why BFS is correct for shortest paths on unweighted graphs but incorrect for weighted graphs?
- Can you implement island-counting on a 2D grid from memory?

If you feel confident, progress to **02_Topological_Sort_and_DSU.ipynb**, where we explore sorting DAGs (Topological Sort) and handling dynamic equivalence classes (Disjoint Set Union).